Retrieve news data per success story.

In [1]:
import requests
import pandas as pd
from io import StringIO

# ── DATA ───────────────────────────────────────────────────────────────────────
# load the cleaned DataFrame
df = pd.read_csv('2_cleaned_microsoft_customers.csv')
# convert the date column to datetime
df['date_cleaned'] = pd.to_datetime(df['date_cleaned'])

# ── ITERATION ──────────────────────────────────────────────────────────────────
# iterate through all rows of the DataFrame
for index, row in df.iterrows():

# ── INPUTS ─────────────────────────────────────────────────────────────────────
    # define the desired modes
    modes       = [
        "timelinetone",
        "timelinevolraw"
    ]

    # extract the company name and use case
    company     = row['company_name_cleaned']
    end_date    = row['date_cleaned'].strftime('%Y%m%d')
    start_date  = (row['date_cleaned'] - pd.DateOffset(days=90)).strftime('%Y%m%d')

    # print the values
    print(f"Company: {company}, Start Date: {start_date}, End Date: {end_date}")

    # ── SETUP ──────────────────────────────────────────────────────────────────
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/115.0.0.0 Safari/537.36"
        )
    }

# ── FETCH, LOAD & MERGE ────────────────────────────────────────────────────────
    # create a DataFrame to hold the merged results
    merged_df = pd.DataFrame()

    # iterate through the modes to merge them
    for mode in modes:
        params = {
            "query":         f"{company}",
            "mode":          mode,
            "format":        "csv",
            "startdatetime": f"{start_date}000000",
            "enddatetime":   f"{end_date}000000",
            "maxrecords":    250,
            "sort":          "datedesc"
        }

        # retrieve response from the API
        resp = requests.get(url, params=params, headers=headers)
        
        # check if the response is successful
        if resp.status_code != 200:
            print(f"Error fetching {mode}: {resp.status_code}")
            continue
        if len(resp.text) <= 1:
            print(f"Empty response for {mode}, skipping…")
            continue

        # read CSV into DataFrame
        resp_df = pd.read_csv(StringIO(resp.text))

        # pivot if it’s one of the serialized timeline modes
        if mode in {"timelinetone", "timelinevolraw", "timelinelang", "timelinesourcecountry"}:
            if {'Date', 'Series', 'Value'}.issubset(resp_df.columns):
                resp_df = resp_df.pivot(index="Date", columns="Series", values="Value").reset_index()
            else:
                print(f"Missing columns in {mode}, skipping…")
                continue

        # merge into previously initialized DataFrame
        if merged_df.empty:
            merged_df = resp_df
        else:
            merged_df = pd.merge(merged_df, resp_df, on="Date", how="outer")

            print(f"Loaded mode={mode}, rows={len(resp_df)}")

# ── CALCULATION ────────────────────────────────────────────────────────────────
            # if you have any missing values, you may want to drop or fill them first:
            merged_df = merged_df.dropna(subset=['Average Tone', 'Article Count', 'Total Monitored Articles'])

            # calculate weighted average of “Average Tone” by “Article Count”
            weighted_tone = (
                (merged_df['Average Tone'] * merged_df['Article Count']).sum()
                / merged_df['Article Count'].sum()
            )

            # calculate weighted average of “Article Count” by “Total Monitored Articles”
            weighted_article_count = (
                (merged_df['Article Count'] / merged_df['Total Monitored Articles']).mean()
            )

            print(f"Weighted average tone: {weighted_tone:.10f}; Weighted article count: {weighted_article_count:.10f}")

# ── SAVE ───────────────────────────────────────────────────────────────────────
            df.loc[index, 'weighted_tone'] = weighted_tone
            df.loc[index, 'weighted_article_count'] = weighted_article_count

df.to_csv('3_news_microsoft_customers.csv', index=False)

Company: Dynea, Start Date: 20220126, End Date: 20220426
Loaded mode=timelinevolraw, rows=91
Weighted average tone: 1.4231947368; Weighted article count: 0.0000006284
Company: DENSO, Start Date: 20240904, End Date: 20241203
Loaded mode=timelinevolraw, rows=91
Weighted average tone: 1.5699917293; Weighted article count: 0.0000172933
Company: Krones, Start Date: 20240817, End Date: 20241115
Loaded mode=timelinevolraw, rows=91
Weighted average tone: 2.1418540984; Weighted article count: 0.0000037916
Company: Brisa, Start Date: 20191218, End Date: 20200317
Loaded mode=timelinevolraw, rows=91
Weighted average tone: 1.6966745614; Weighted article count: 0.0000029598
Company: Pajson, Start Date: 20210902, End Date: 20211201
Empty response for timelinetone, skipping…
Empty response for timelinevolraw, skipping…
Company: Schaeffler, Start Date: 20240702, End Date: 20240930
Loaded mode=timelinevolraw, rows=91
Weighted average tone: 1.0687769784; Weighted article count: 0.0000088955
Company: Pira

In [ ]:
import requests
import pandas as pd
from io import StringIO

# ── DATA ───────────────────────────────────────────────────────────────────────
# load the cleaned DataFrame
df = pd.read_csv('2_cleaned_microsoft_customers.csv')
# convert the date column to datetime
df['date_cleaned'] = pd.to_datetime(df['date_cleaned'])

# ── ITERATION ──────────────────────────────────────────────────────────────────
# iterate through all rows of the DataFrame
for index, row in df.iterrows():

    # ── INPUTS FOR PAST 90 DAYS ───────────────────────────────────────────────
    company       = row['company_name_cleaned']
    end_date_past = row['date_cleaned'].strftime('%Y%m%d')
    start_date_past = (row['date_cleaned'] - pd.DateOffset(days=90)).strftime('%Y%m%d')

    print(f"[PAST 90 DAYS] Company: {company}, Start Date: {start_date_past}, End Date: {end_date_past}")

    # ── FETCH & CALCULATE FOR PAST 90 DAYS ────────────────────────────────────
    # modes to fetch
    modes = ["timelinetone", "timelinevolraw"]
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/115.0.0.0 Safari/537.36"
        )
    }

    merged_df_past = pd.DataFrame()
    for mode in modes:
        params = {
            "query":         f"{company}",
            "mode":          mode,
            "format":        "csv",
            "startdatetime": f"{start_date_past}000000",
            "enddatetime":   f"{end_date_past}000000",
            "maxrecords":    250,
            "sort":          "datedesc"
        }

        resp = requests.get(url, params=params, headers=headers)
        if resp.status_code != 200:
            print(f"  Error fetching {mode} for past 90 days: {resp.status_code}")
            continue
        if len(resp.text) <= 1:
            print(f"  Empty response for {mode} (past), skipping…")
            continue

        resp_df = pd.read_csv(StringIO(resp.text))
        if mode in {"timelinetone", "timelinevolraw", "timelinelang", "timelinesourcecountry"}:
            if {'Date', 'Series', 'Value'}.issubset(resp_df.columns):
                resp_df = resp_df.pivot(index="Date", columns="Series", values="Value").reset_index()
            else:
                print(f"  Missing columns in {mode} (past), skipping…")
                continue

        if merged_df_past.empty:
            merged_df_past = resp_df
        else:
            merged_df_past = pd.merge(merged_df_past, resp_df, on="Date", how="outer")
        print(f"  Loaded mode={mode} (past), rows={len(resp_df)}")

    # Only calculate if we have data
    if not merged_df_past.empty:
        merged_df_past = merged_df_past.dropna(subset=['Average Tone', 'Article Count', 'Total Monitored Articles'])

        weighted_tone_past = (
            (merged_df_past['Average Tone'] * merged_df_past['Article Count']).sum()
            / merged_df_past['Article Count'].sum()
        )
        weighted_article_count_past = (
            (merged_df_past['Article Count'] / merged_df_past['Total Monitored Articles']).mean()
        )

        print(f"  [PAST 90 DAYS] Weighted average tone: {weighted_tone_past:.10f}; "
              f"Weighted article count: {weighted_article_count_past:.10f}")
    else:
        weighted_tone_past = None
        weighted_article_count_past = None
        print("  No data for past 90 days; setting metrics to None.")

    df.loc[index, 'weighted_tone'] = weighted_tone_past
    df.loc[index, 'weighted_article_count'] = weighted_article_count_past

    # ── INPUTS FOR NEXT 90 DAYS ────────────────────────────────────────────────
    # Define the next-90-days window (starting the day after `date_cleaned`)
    start_date_next = (row['date_cleaned'] + pd.DateOffset(days=1)).strftime('%Y%m%d')
    end_date_next   = (row['date_cleaned'] + pd.DateOffset(days=90)).strftime('%Y%m%d')

    print(f"[NEXT 90 DAYS] Company: {company}, Start Date: {start_date_next}, End Date: {end_date_next}")

    # ── FETCH & CALCULATE FOR NEXT 90 DAYS ────────────────────────────────────
    merged_df_next = pd.DataFrame()
    for mode in modes:
        params = {
            "query":         f"{company}",
            "mode":          mode,
            "format":        "csv",
            "startdatetime": f"{start_date_next}000000",
            "enddatetime":   f"{end_date_next}000000",
            "maxrecords":    250,
            "sort":          "datedesc"
        }

        resp = requests.get(url, params=params, headers=headers)
        if resp.status_code != 200:
            print(f"  Error fetching {mode} for next 90 days: {resp.status_code}")
            continue
        if len(resp.text) <= 1:
            print(f"  Empty response for {mode} (next), skipping…")
            continue

        resp_df = pd.read_csv(StringIO(resp.text))
        if mode in {"timelinetone", "timelinevolraw", "timelinelang", "timelinesourcecountry"}:
            if {'Date', 'Series', 'Value'}.issubset(resp_df.columns):
                resp_df = resp_df.pivot(index="Date", columns="Series", values="Value").reset_index()
            else:
                print(f"  Missing columns in {mode} (next), skipping…")
                continue

        if merged_df_next.empty:
            merged_df_next = resp_df
        else:
            merged_df_next = pd.merge(merged_df_next, resp_df, on="Date", how="outer")
        print(f"  Loaded mode={mode} (next), rows={len(resp_df)}")

    # Only calculate if we have data
    if not merged_df_next.empty:
        merged_df_next = merged_df_next.dropna(subset=['Average Tone', 'Article Count', 'Total Monitored Articles'])

        weighted_tone_next = (
            (merged_df_next['Average Tone'] * merged_df_next['Article Count']).sum()
            / merged_df_next['Article Count'].sum()
        )
        weighted_article_count_next = (
            (merged_df_next['Article Count'] / merged_df_next['Total Monitored Articles']).mean()
        )

        print(f"  [NEXT 90 DAYS] Weighted average tone: {weighted_tone_next:.10f}; "
              f"Weighted article count: {weighted_article_count_next:.10f}")
    else:
        weighted_tone_next = None
        weighted_article_count_next = None
        print("  No data for next 90 days; setting metrics to None.")

    df.loc[index, 'weighted_tone_next90'] = weighted_tone_next
    df.loc[index, 'weighted_article_count_next90'] = weighted_article_count_next

# ── SAVE ───────────────────────────────────────────────────────────────────────
df.to_csv('3_news_microsoft_customers.csv', index=False)
